In [113]:
#conect with drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "xx"
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"] = "xx"

In [115]:
#!sed -i 's/\r$//' /content/drive/MyDrive/apai/fetch_src.sh  # Windows users
!bash /content/drive/MyDrive/apai/fetch_src.sh

 fetch_src.sh — Download /src from GitHub

Checking environment...
GITHUB_TOKEN set
Fetching file list from GitHub API...
Found 17 file(s) under /src/
src/cl_strategies/ewc.py -> /content/src/cl_strategies/ewc.py
src/cl_strategies/reharsal.py -> /content/src/cl_strategies/reharsal.py
src/config/config.py -> /content/src/config/config.py
src/data/dataset.py -> /content/src/data/dataset.py
src/data/preprocessing.py -> /content/src/data/preprocessing.py
src/data/study_dataset.py -> /content/src/data/study_dataset.py
src/fine_tune/trainer.py -> /content/src/fine_tune/trainer.py
src/fine_tune/visualizer.py -> /content/src/fine_tune/visualizer.py
src/imports.py -> /content/src/imports.py
src/kd/mseLoss.py -> /content/src/kd/mseLoss.py
src/kd/utils.py -> /content/src/kd/utils.py
src/meta_learning/models.py -> /content/src/meta_learning/models.py
src/meta_learning/reptile.py -> /content/src/meta_learning/reptile.py
src/meta_learning/sampler.py -> /content/src/meta_learning/sampler.py
src/model

In [116]:
#!sed -i 's/\r$//' /content/drive/MyDrive/apai/setup_colab.sh    # Windows users
!bash /content/drive/MyDrive/apai/setup_colab.sh


UCF101 Project — Colab Setup Script

Checking environment...
KAGGLE_USERNAME set
KAGGLE_KEY set
config.py found

Installing kagglehub...
Done.

Using Colab cache for faster access to the 'ucf101-action-recognition' dataset.
Dataset path: /kaggle/input/ucf101-action-recognition

Detecting path and updating config.py...
Found in Kaggle input: /kaggle/input/ucf101-action-recognition/
DATASET_ROOT = '/kaggle/input/ucf101-action-recognition/'
OUTPUT_ROOT  = '/kaggle/working/ucf101-processed/'

Setup complete!
config.py updated on Drive.
Restarting kernel in 5 seconds...
Dataset cache stays on disk.
Run Cell 3 after restart to verify.
Restarting in 5...
Restarting in 4...
Restarting in 3...
Restarting in 2...
Restarting in 1...


In [117]:
# imports
%run /content/src/imports.py

Using device: cuda


In [ ]:
preprocess_dataset(
    target_classes=cfg.SELECTED_CLASSES,
    output_root=cfg.EXEMPLAR_ROOT,
    max_samples=cfg.EXEMPLAR_LIMIT
)


--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_exemplars
Classes: 10 | Limit: 20

Processing split: train



Processing split: val



Processing split: test



Done! Dataset ready at: ./UCF101/processed_exemplars


In [ ]:
pin_memory = device.type == "cuda"

Device initialized: cuda


In [120]:
old_classes = cfg.SELECTED_CLASSES
num_old = len(old_classes)
print(f"Selected classes ({num_old}): {old_classes}")

c2i = {c: i for i, c in enumerate(old_classes)}
old_train = UCF101Clips(f"{cfg.EXEMPLAR_ROOT}/train", class_to_idx=c2i)
old_val   = UCF101Clips(f"{cfg.EXEMPLAR_ROOT}/val",   class_to_idx=c2i)
old_test  = UCF101Clips(f"{cfg.EXEMPLAR_ROOT}/test",  class_to_idx=c2i)

train_loader = make_loader(old_train, batch_size=cfg.BATCH_SIZE, num_workers=cfg.NUM_WORKERS, pin_memory=pin_memory)
val_loader   = make_loader(old_val,   batch_size=cfg.BATCH_SIZE, num_workers=cfg.NUM_WORKERS, pin_memory=pin_memory)
test_loader  = make_loader(old_test,  batch_size=cfg.BATCH_SIZE, num_workers=cfg.NUM_WORKERS, pin_memory=pin_memory)

print(f"Train: {len(old_train)} | Val: {len(old_val)} | Test: {len(old_test)}")

Selected classes (10): ['PlayingTabla', 'PommelHorse', 'JumpingJack', 'PushUps', 'PoleVault', 'HorseRace', 'HighJump', 'Drumming', 'HorseRiding', 'Diving']
Train: 200 | Val: 165 | Test: 171


In [169]:
teacher = ResNet50LSTMTeacher(hidden_size=cfg.TEACHER_HIDDEN, num_classes=num_old, dropout_p=0.4).to(device)
student_base = MobileNetV3SmallLSTMStudent(num_classes=num_old, hidden_size=cfg.STUDENT_HIDDEN, dropout_p=cfg.STUDENT_DROPOUT).to(device)
student_meta = MobileNetV3SmallLSTMStudent(num_classes=num_old, hidden_size=cfg.STUDENT_HIDDEN, dropout_p=cfg.STUDENT_DROPOUT).to(device)
student_std  = MobileNetV3SmallLSTMStudent(num_classes=num_old, hidden_size=cfg.STUDENT_HIDDEN, dropout_p=cfg.STUDENT_DROPOUT).to(device)

raw_ckpt = torch.load(cfg.RESNET50_PATH, map_location=device)
mapped_ckpt = remap_teacher_checkpoint(raw_ckpt)
missing, unexpected = teacher.load_state_dict(mapped_ckpt, strict=False)

print(f"Teacher weights loaded ({len(mapped_ckpt)} keys).")

teacher_class_acc, teacher_total_acc = calculate_accuracies(teacher, test_loader, num_old, device)
student_base_class_acc, student_base_total_acc = calculate_accuracies(student_base, test_loader, num_old, device)

print(f"  Teacher Test Accuracy     : {teacher_total_acc:.2f}%")
print(f"  Raw Student Test Accuracy : {student_base_total_acc:.2f}%")

  Dropped 53 num_batches_tracked buffer(s).
Teacher weights loaded (271 keys).
  Teacher Test Accuracy     : 97.08%
  Raw Student Test Accuracy : 6.43%


In [ ]:
train_t_embs, train_labels = extract_embeddings(teacher, train_loader, device)
val_t_embs,   val_labels   = extract_embeddings(teacher, val_loader, device)
test_t_embs,  test_labels  = extract_embeddings(teacher, test_loader, device)

train_s_meta, _ = extract_embeddings(student_meta, train_loader, device)
val_s_meta,   _ = extract_embeddings(student_meta, val_loader, device)
test_s_meta,  _ = extract_embeddings(student_meta, test_loader, device)

train_s_std, _ = extract_embeddings(student_std, train_loader, device)
val_s_std,   _ = extract_embeddings(student_std, val_loader, device)
test_s_std,  _ = extract_embeddings(student_std, test_loader, device)


Extracting teacher embeddings

Extracting META student embeddings

Extracting STD student embeddings


In [157]:
# Optional safety cap to guarantee 5-shot.
train_s_meta_5, train_t_meta_5, train_y_meta_5 = limit_to_k_per_class(
    train_s_meta, train_t_embs, train_labels, k_per_class=5, seed=cfg.SEED
)
train_s_std_5, train_t_std_5, train_y_std_5 = limit_to_k_per_class(
    train_s_std, train_t_embs, train_labels, k_per_class=5, seed=cfg.SEED
)

val_s_meta_use, val_y_meta_use = val_s_meta, val_labels
val_s_std_use,  val_y_std_use  = val_s_std, val_labels

In [158]:
print("Embeddings count")
print(f"  META student train embeddings (5-shot): {train_s_meta_5.shape[0]}")
print(f"  STD student train embeddings (5-shot): {train_s_std_5.shape[0]}")
print(f"\n  META student train embeddings (full): {train_t_meta_5.shape[0]}")


Embeddings count
  META student train embeddings (5-shot): 50
  STD student train embeddings (5-shot): 50

  META student train embeddings (full): 50


## **WITH MAMAL**

In [159]:
head_meta = EmbeddingHead(cfg.STUDENT_HIDDEN, cfg.TEACHER_HIDDEN, num_classes=num_old, dropout_p=cfg.HEAD_DROPOUT).to(device)

head_meta = train_reptile(
    head_meta,
    train_s_meta_5, train_t_meta_5, train_y_meta_5,
    val_s_meta_use, val_y_meta_use, num_old, device,
    cfg.REPTILE_EPOCHS, cfg.EPISODES_PER_EPOCH,
    cfg.K_SUPPORT, cfg.K_QUERY, cfg.INNER_LR, cfg.INNER_STEPS,
    cfg.SRC_EPSILON, cfg.CE_WEIGHT, cfg.MSE_WEIGHT, cfg.REPTILE_PATIENCE,
)

head_meta = finetune_head(
    head_meta,
    train_s_meta_5, train_t_meta_5, train_y_meta_5,
    val_s_meta_use, val_y_meta_use, device,
    cfg.FINETUNE_EPOCHS, cfg.FINETUNE_LR, cfg.FINETUNE_WD,
    cfg.FINETUNE_BATCH, cfg.FINETUNE_PATIENCE, cfg.CE_WEIGHT, cfg.MSE_WEIGHT,
    label_smoothing=cfg.LABEL_SMOOTHING,
)

meta_head_val_acc  = eval_head(head_meta, val_s_meta_use, val_y_meta_use, device)
meta_head_test_acc = eval_head(head_meta, test_s_meta, test_labels, device)

student_meta = restore_head_to_student(student_meta, head_meta)
compare_head_vs_student_on_same_batch(student_meta, head_meta, test_loader, device)
meta_class_acc, meta_total_acc = calculate_accuracies(student_meta, test_loader, num_old, device)

print(f"\n  Meta — val: {meta_head_val_acc*100:.2f}%  test(emb): {meta_head_test_acc*100:.2f}%  test(frames): {meta_total_acc:.2f}%")


[Reptile] Epoch 01/25 | Train: 14.00% | Val: 12.73% | Loss: 2.4511
[Reptile] Epoch 02/25 | Train: 20.00% | Val: 15.76% | Loss: 2.3481
[Reptile] Epoch 03/25 | Train: 26.00% | Val: 21.82% | Loss: 2.3872
[Reptile] Epoch 04/25 | Train: 42.00% | Val: 25.45% | Loss: 2.0803
[Reptile] Epoch 05/25 | Train: 60.00% | Val: 26.06% | Loss: 2.0073
[Reptile] Epoch 06/25 | Train: 72.00% | Val: 29.09% | Loss: 1.9811
[Reptile] Epoch 07/25 | Train: 84.00% | Val: 31.52% | Loss: 1.8501
[Reptile] Epoch 08/25 | Train: 86.00% | Val: 38.79% | Loss: 1.8109
[Reptile] Epoch 09/25 | Train: 88.00% | Val: 43.64% | Loss: 1.6470
[Reptile] Epoch 10/25 | Train: 90.00% | Val: 46.06% | Loss: 1.5894
[Reptile] Epoch 11/25 | Train: 94.00% | Val: 50.91% | Loss: 1.5287
[Reptile] Epoch 12/25 | Train: 94.00% | Val: 52.73% | Loss: 1.5066
[Reptile] Epoch 13/25 | Train: 94.00% | Val: 55.15% | Loss: 1.5254
[Reptile] Epoch 14/25 | Train: 94.00% | Val: 55.15% | Loss: 1.4341
[Reptile] Epoch 15/25 | Train: 98.00% | Val: 55.76% | Loss: 1.

## **NO-MAMAL**

In [160]:
head_std = EmbeddingHead(cfg.STUDENT_HIDDEN, cfg.TEACHER_HIDDEN, num_classes=num_old, dropout_p=cfg.HEAD_DROPOUT).to(device)

head_std = finetune_head(
    head_std,
    train_s_std_5, train_t_std_5, train_y_std_5,
    val_s_std_use, val_y_std_use, device,
    cfg.FINETUNE_EPOCHS, cfg.FINETUNE_LR, cfg.FINETUNE_WD,
    cfg.FINETUNE_BATCH, cfg.FINETUNE_PATIENCE, cfg.CE_WEIGHT, cfg.MSE_WEIGHT,
    label_smoothing=cfg.LABEL_SMOOTHING,
)

std_head_val_acc  = eval_head(head_std, val_s_std_use, val_y_std_use, device)
std_head_test_acc = eval_head(head_std, test_s_std, test_labels, device)

student_std = restore_head_to_student(student_std, head_std)
compare_head_vs_student_on_same_batch(student_std, head_std, test_loader, device)
std_class_acc, std_total_acc = calculate_accuracies(student_std, test_loader, num_old, device)

print(f"\n  Std — val: {std_head_val_acc*100:.2f}%  test(emb): {std_head_test_acc*100:.2f}%  test(frames): {std_total_acc:.2f}%")

[FineTune] Epoch 01/15 | Train: 20.00% | Val: 11.52% | Loss: 2.9771


[FineTune] Epoch 02/15 | Train: 24.00% | Val: 12.73% | Loss: 2.2052
[FineTune] Epoch 03/15 | Train: 26.00% | Val: 12.73% | Loss: 2.2352
[FineTune] Epoch 04/15 | Train: 26.00% | Val: 15.15% | Loss: 2.1619
[FineTune] Epoch 05/15 | Train: 28.00% | Val: 16.36% | Loss: 2.0352
[FineTune] Epoch 06/15 | Train: 30.00% | Val: 18.18% | Loss: 1.6920
[FineTune] Epoch 07/15 | Train: 32.00% | Val: 19.39% | Loss: 2.7068
[FineTune] Epoch 08/15 | Train: 32.00% | Val: 19.39% | Loss: 2.5358
[FineTune] Epoch 09/15 | Train: 32.00% | Val: 19.39% | Loss: 2.7133
[FineTune] Epoch 10/15 | Train: 34.00% | Val: 20.61% | Loss: 2.1869
[FineTune] Epoch 11/15 | Train: 34.00% | Val: 20.00% | Loss: 2.0426
[FineTune] Epoch 12/15 | Train: 40.00% | Val: 20.00% | Loss: 1.8803
[FineTune] Epoch 13/15 | Train: 40.00% | Val: 20.61% | Loss: 1.8956
[FineTune] Epoch 14/15 | Train: 40.00% | Val: 20.61% | Loss: 2.0463
[FineTune] Epoch 15/15 | Train: 42.00% | Val: 21.21% | Loss: 1.7085
  -> Best fine-tune val acc: 21.21%
  Restored: 

In [166]:
print(f"{'Class':<28} | {'Teacher':>10} | {'Raw Stud':>10} | {'Meta Stud':>10} | {'Std Stud':>10}")
for idx, name in enumerate(old_classes):
    print(f"{name:<28} | "
          f"{teacher_class_acc[idx]:>9.2f}% | "
          f"{student_base_class_acc[idx]:>9.2f}% | "
          f"{meta_class_acc[idx]:>9.2f}% | "
              f"{std_class_acc[idx]:>9.2f}%")

Class                        |    Teacher |   Raw Stud |  Meta Stud |   Std Stud
PlayingTabla                 |    100.00% |     50.00% |     71.43% |     71.43%
PommelHorse                  |    100.00% |      0.00% |     68.75% |     25.00%
JumpingJack                  |    100.00% |     43.75% |     68.75% |      6.25%
PushUps                      |    100.00% |      0.00% |     69.23% |     23.08%
PoleVault                    |     89.47% |      0.00% |     73.68% |      0.00%
HorseRace                    |    100.00% |      0.00% |     50.00% |      0.00%
HighJump                     |     87.50% |     50.00% |     50.00% |     18.75%
Drumming                     |    100.00% |      0.00% |     80.95% |     76.19%
HorseRiding                  |    100.00% |      0.00% |     42.86% |      0.00%
Diving                       |     94.74% |      0.00% |     52.63% |     21.05%


In [167]:
print(f"{'Class':<28} | {'Teacher':>10} | {'Raw Stud':>10} | {'Meta Stud':>10} | {'Std Stud':>10}")
print(f"{'OVERALL':<28} | "
      f"{teacher_total_acc:>9.2f}% | "
      f"{student_base_total_acc:>9.2f}% | "
      f"{meta_total_acc:>9.2f}% | "
      f"{std_total_acc:>9.2f}%")

Class                        |    Teacher |   Raw Stud |  Meta Stud |   Std Stud
OVERALL                      |     97.08% |     12.87% |     62.57% |     23.98%


In [168]:
wrapped_student_meta = ModelWrapper(student_meta)

student_meta_results = evaluate_all_tasks(
    model=wrapped_student_meta,
    device=device,
    base_loader=val_loader,
    task_loaders=[val_loader],
    combined_loaders=[test_loader]
)

  base             -> Acc=0.6303  Loss=1.5491
  task1_only       -> Acc=0.6303  Loss=1.5491
  combined_t1      -> Acc=0.6257  Loss=1.5336
